# NB4 — Timing-Based DAG Construction (v2 Final)

## What changed from v1 and why

The original used `start_j >= end_i` as edge condition — **time order ≠ dependency**.
Even tiny scheduling gaps enforced edges, serializing graphs and killing parallelism.
The model then learned durations instead of topology → near-zero Spearman ρ (~0.03).

### v2 fixes (in order of impact):
| Fix | Impact on ρ |
|-----|-------------|
| EPS tolerance — ignore tiny timing gaps | ★★★ Removes false serialization |
| MAX_PARENTS pruning — cap incoming edges | ★★★ Forces cleaner fork-join structure |
| Parallel overlap detection | ★★ Concurrent ops not chained |
| Random edge dropout (15%) | ★★ Breaks residual false deps |
| Critical path ratio filter (>80%) | ★★★ Drops useless chain graphs from training |
| Graph quality metrics printed before training | Validates fixes worked |

### v2 Final additional fixes:
| Fix | Why |
|-----|-----|
| `y_step` stored as proper tensor shape `[1]` | Prevents nb3 shape mismatch |
| Edge index validation (no self-loops, within bounds) | Prevents PyG errors |
| Synthetic augmentation for serial trace fallback | If CP ratio still > 0.7 after fixes |
| `load_ops_from_trace` handles both list and dict JSON | Broader trace compatibility |

In [ ]:
import os
import json
import random
import pickle
import shutil
import numpy as np
import networkx as nx
import torch
from torch_geometric.data import HeteroData
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

print('Libraries loaded.')

## Configuration

Tune `EPS_FRACTION` and `MAX_PARENTS` first — these two have the largest structural impact.
Check the quality report before training: if `avg_cp_ratio > 0.80`, lower `EPS_FRACTION`.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
TRACE_DIR = './traces'
DRIVE_MOUNT_POINT = '/content/drive'
DRIVE_FOLDER = 'MyDrive/BottleneckOracle_Backup'

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT_POINT)
    IN_COLAB = True
except Exception:
    print('Colab Drive not available; using local workspace paths.')

if IN_COLAB:
    DRIVE_BASE = os.path.join(DRIVE_MOUNT_POINT, DRIVE_FOLDER)
else:
    DRIVE_BASE = '.'

os.makedirs(DRIVE_BASE, exist_ok=True)

OUTPUT_PATH = os.path.join(DRIVE_BASE, 'graphs_v2.pkl')
GRAPH_QUALITY_FIG_PATH = os.path.join(DRIVE_BASE, 'graph_quality_v2.png')
SAMPLE_DAG_FIG_PATH = os.path.join(DRIVE_BASE, 'sample_dag_v2.png')
BACKUP_ZIP_PATH = os.path.join(DRIVE_BASE, 'nb4_data_backup.zip')

print(f'Dataset output path: {OUTPUT_PATH}')

# ── FIX 1: EPS tolerance ───────────────────────────────────────────────────────
# Relax strict start_j >= end_i → require start_j >= end_i + EPS
# Small scheduling gaps (async dispatch, rounding) no longer enforce edges.
EPS_FRACTION = 0.05   # EPS = EPS_FRACTION * avg_duration of the graph

# ── FIX 2: MAX_PARENTS pruning ─────────────────────────────────────────────────
# Keep only the most recent (temporally closest) MAX_PARENTS parents per node.
# Forces cleaner fork-join structure, removes over-connection.
MAX_PARENTS = 2

# ── FIX 3: Parallel overlap threshold ─────────────────────────────────────────
# If two ops overlap by > OVERLAP_THRESHOLD fraction of the shorter op → parallel,
# skip the edge entirely.
OVERLAP_THRESHOLD = 0.3

# ── FIX 4: Random edge dropout ────────────────────────────────────────────────
# Drop a small fraction of candidate edges to break residual false dependencies.
EDGE_DROP_PROB = 0.10

# ── FIX 5: Critical path ratio filter ─────────────────────────────────────────
# Discard graphs where critical_path_length / total_duration is too large.
# Such graphs are near-chains and add little ranking signal.
CP_THRESHOLD = 0.80
MAX_CP_RATIO = CP_THRESHOLD
MIN_NODES    = 10

print('Configuration set.')
print(f'  EPS_FRACTION  = {EPS_FRACTION}')
print(f'  MAX_PARENTS   = {MAX_PARENTS}')
print(f'  OVERLAP_THR   = {OVERLAP_THRESHOLD}')
print(f'  EDGE_DROP     = {EDGE_DROP_PROB}')
print(f'  CP_THRESHOLD  = {CP_THRESHOLD}')

In [ ]:
# ── Restore traces from Drive backup ─────────────────────────────────────────────
ZIP_PATH = os.path.join(DRIVE_BASE, 'traces_backup.zip')
EXTRACT_DIR = TRACE_DIR

os.makedirs(EXTRACT_DIR, exist_ok=True)

print(f"Looking for trace backup at: {ZIP_PATH}")
print(f"Extracting into: {EXTRACT_DIR}")

if os.path.exists(ZIP_PATH):
    shutil.unpack_archive(ZIP_PATH, EXTRACT_DIR)
    trace_files = [f for f in os.listdir(EXTRACT_DIR) if f.endswith('.json')]
    print(f"✅ Extraction complete! Found {len(trace_files)} trace files.")
else:
    print(f"❌ Error: Couldn't find the zip file at {ZIP_PATH}. Check the file name and path!")

## Core DAG Builder

Replaces the original strict-ordering approach with all five structural fixes.

In [ ]:
def compute_overlap_fraction(start_i, end_i, start_j, end_j):
    """Return overlap / min(dur_i, dur_j). Used for FIX 3 parallel detection."""
    overlap_start = max(start_i, start_j)
    overlap_end   = min(end_i, end_j)
    overlap       = max(0.0, overlap_end - overlap_start)
    min_dur       = min(end_i - start_i, end_j - start_j)
    if min_dur <= 0:
        return 0.0
    return overlap / min_dur


def build_dag_from_ops(ops):
    """
    Build a DAG from a list of operation dicts.
    Each op must have: 'name', 'start_time', 'duration'

    Applies all 5 structural fixes.
    Returns nx.DiGraph or None if degenerate.
    """
    n = len(ops)
    if n < MIN_NODES:
        return None

    # Precompute end times
    for op in ops:
        op['end_time'] = op['start_time'] + op['duration']

    # FIX 1: EPS based on average duration of this graph
    avg_dur = np.mean([op['duration'] for op in ops])
    EPS = EPS_FRACTION * max(avg_dur, 1e-9)

    # Build candidate edges
    candidate_edges = []

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            si, ei = ops[i]['start_time'], ops[i]['end_time']
            sj, ej = ops[j]['start_time'], ops[j]['end_time']

            # FIX 3: skip parallel ops (significant overlap)
            if compute_overlap_fraction(si, ei, sj, ej) >= OVERLAP_THRESHOLD:
                continue

            # FIX 1: j only depends on i if j starts meaningfully after i ends
            if sj >= ei + EPS:
                # FIX 4: random edge dropout
                if random.random() < EDGE_DROP_PROB:
                    continue
                candidate_edges.append((i, j))

    # FIX 2: MAX_PARENTS pruning — keep only closest-in-time parents per node
    parents_of = defaultdict(list)  # j → list of (end_time_of_i, i)
    for (i, j) in candidate_edges:
        parents_of[j].append((ops[i]['end_time'], i))

    pruned_edges = []
    for j, parent_list in parents_of.items():
        parent_list.sort(key=lambda x: x[0], reverse=True)  # most recent first
        for (_, i) in parent_list[:MAX_PARENTS]:
            pruned_edges.append((i, j))

    # Build NetworkX DiGraph
    G = nx.DiGraph()
    for idx, op in enumerate(ops):
        G.add_node(idx,
                   name=op.get('name', f'op_{idx}'),
                   start_time=op['start_time'],
                   end_time=op['end_time'],
                   duration=op['duration'])

    for (i, j) in pruned_edges:
        G.add_edge(i, j)

    # Reject degenerate near-chains that never branch.
    if n > 5 and all(G.in_degree(node) <= 1 for node in G.nodes()):
        return None

    # Ensure DAG (remove back-edges from any accidental cycles)
    if not nx.is_directed_acyclic_graph(G):
        for cycle in list(nx.simple_cycles(G)):
            if len(cycle) > 1:
                G.remove_edge(cycle[-1], cycle[0])
        if not nx.is_directed_acyclic_graph(G):
            return None

    return G


print('DAG builder defined.')

## Critical Path & Slack Computation (CPM)

Uses Critical Path Method forward + backward pass to compute per-node slack.
This is the learning target — NOT leaked as a feature.

In [ ]:
def compute_cpm_slack(G):
    """
    CPM forward (EFT) + backward (LFT) pass.
    Returns dict: node_id → float slack (= LFT - EFT, non-negative).
    Nodes with slack=0 lie on the critical path.
    """
    topo = list(nx.topological_sort(G))

    # Forward pass: Earliest Finish Time (EFT)
    EFT = {}
    for node in topo:
        dur = G.nodes[node]['duration']
        if G.in_degree(node) == 0:
            EFT[node] = G.nodes[node]['start_time'] + dur
        else:
            EFT[node] = max(EFT[p] for p in G.predecessors(node)) + dur

    project_end = max(EFT.values())

    # Backward pass: Latest Finish Time (LFT)
    LFT = {}
    for node in reversed(topo):
        if G.out_degree(node) == 0:
            LFT[node] = project_end
        else:
            LFT[node] = (
                min(LFT[s] - G.nodes[s]['duration'] for s in G.successors(node))
                + G.nodes[node]['duration']
            )

    # Slack = LFT - EFT (clamp at 0)
    slack = {node: max(0.0, LFT[node] - EFT[node]) for node in G.nodes()}
    return slack


def compute_cp_ratio(G, slack):
    """Critical path duration divided by total task duration."""
    topo = list(nx.topological_sort(G))
    eft = {}
    for node in topo:
        dur = G.nodes[node]['duration']
        if G.in_degree(node) == 0:
            eft[node] = dur
        else:
            eft[node] = max(eft[p] for p in G.predecessors(node)) + dur
    critical_path_length = max(eft.values()) if eft else 0.0
    total_duration = sum(G.nodes[node]['duration'] for node in G.nodes())
    return critical_path_length / max(total_duration, 1e-9)


print('CPM slack computation defined.')

## Graph Quality Metrics

Always run this before training. Targets:
- `avg_cp_ratio < 0.80`
- `avg_branching_factor > 1.30`

In [ ]:
def graph_quality_report(graphs_with_slack):
    """
    graphs_with_slack: list of (G, slack_dict)
    Prints quality report and returns summary dict.
    """
    cp_ratios, branching_factors, slack_variances, node_counts = [], [], [], []

    for G, slack in graphs_with_slack:
        cp_ratios.append(compute_cp_ratio(G, slack))
        out_degs = [G.out_degree(n) for n in G.nodes()]
        branching_factors.append(np.mean(out_degs))
        slack_variances.append(np.var(list(slack.values())))
        node_counts.append(len(G.nodes()))

    summary = {
        'n_graphs'            : len(graphs_with_slack),
        'avg_cp_ratio'        : np.mean(cp_ratios),
        'avg_branching_factor': np.mean(branching_factors),
        'avg_slack_variance'  : np.mean(slack_variances),
        'avg_nodes'           : np.mean(node_counts),
    }

    print('=== Graph Quality Report ===')
    print(f"  Graphs kept         : {summary['n_graphs']}")
    print(f"  Avg CP ratio        : {summary['avg_cp_ratio']:.3f}  (target < {CP_THRESHOLD:.2f})")
    print(f"  Avg branching factor: {summary['avg_branching_factor']:.3f}  (target > 1.30)")
    print(f"  Avg slack variance  : {summary['avg_slack_variance']:.6f}")
    print(f"  Avg nodes per graph : {summary['avg_nodes']:.1f}")

    # Quality warnings
    if summary['avg_cp_ratio'] > CP_THRESHOLD:
        print(f'  ⚠ WARNING: avg_cp_ratio > {CP_THRESHOLD:.2f} — graphs are too serial.')
        print('             Lower EPS_FRACTION (try 0.08–0.10) or MAX_PARENTS=1.')
    if summary['avg_branching_factor'] < 1.30:
        print('  ⚠ WARNING: branching_factor < 1.30 — too linear.')
        print('             Increase EDGE_DROP_PROB or reduce MAX_PARENTS.')
    if summary['n_graphs'] < 400:
        print('  ⚠ WARNING: < 400 graphs. Consider synthetic augmentation (see below).')

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    axes[0].hist(cp_ratios, bins=20, color='tomato', edgecolor='white')
    axes[0].axvline(CP_THRESHOLD, color='k', linestyle='--', label='filter threshold')
    axes[0].set_title('CP Ratio'); axes[0].set_xlabel('CP ratio'); axes[0].legend()

    axes[1].hist(branching_factors, bins=20, color='steelblue', edgecolor='white')
    axes[1].axvline(1.3, color='k', linestyle='--', label='good threshold')
    axes[1].set_title('Branching Factor'); axes[1].legend()

    axes[2].hist(slack_variances, bins=20, color='seagreen', edgecolor='white')
    axes[2].set_title('Slack Variance')

    plt.tight_layout()
    plt.savefig(GRAPH_QUALITY_FIG_PATH, dpi=120)
    plt.show()

    return summary


print('Quality metrics defined.')

## HeteroData Conversion

Node features (6 total, **NO slack** — that is the label):
1. `duration` (normalized per-graph)
2. `start_time` (normalized per-graph)
3. `in_degree` (normalized)
4. `out_degree` (normalized)
5. `is_source` (binary)
6. `is_sink` (binary)

In [ ]:
def graph_to_heterodata(G, slack):
    """
    Convert (nx.DiGraph, slack_dict) → torch_geometric HeteroData.
    Node type: 'op'
    Edge type: ('op', 'depends_on', 'op')

    FIX (v2 final):
    - edge_index validated (no self-loops, within bounds)
    - y_step stored as shape [1] float tensor (prevents nb3 shape mismatch)
    """
    nodes = list(G.nodes())
    node_to_idx = {n: i for i, n in enumerate(nodes)}
    N = len(nodes)

    durations   = np.array([G.nodes[n]['duration']   for n in nodes], dtype=np.float32)
    start_times = np.array([G.nodes[n]['start_time'] for n in nodes], dtype=np.float32)
    in_degrees  = np.array([G.in_degree(n)           for n in nodes], dtype=np.float32)
    out_degrees = np.array([G.out_degree(n)          for n in nodes], dtype=np.float32)
    is_source   = (in_degrees  == 0).astype(np.float32)
    is_sink     = (out_degrees == 0).astype(np.float32)

    # Per-graph normalization
    dur_max  = max(durations.max(),   1e-9)
    time_max = max(start_times.max(), 1e-9)
    durations_norm   = durations   / dur_max
    start_times_norm = start_times / time_max

    # Feature matrix — shape (N, 6), NO slack (label leakage would be here)
    x = np.stack([
        durations_norm,
        start_times_norm,
        in_degrees  / max(in_degrees.max(),  1),
        out_degrees / max(out_degrees.max(), 1),
        is_source,
        is_sink,
    ], axis=1)  # (N, 6)

    # Slack labels — continuous CPM slack, NOT leaked as feature
    slack_vals = np.array([slack[n] for n in nodes], dtype=np.float32)

    # Step time: total makespan
    project_end   = max(G.nodes[n]['end_time']   for n in nodes)
    project_start = min(G.nodes[n]['start_time'] for n in nodes)
    step_time     = float(project_end - project_start)

    # Edge index — FIX: validate no self-loops and within bounds
    edges = [(node_to_idx[u], node_to_idx[v]) for u, v in G.edges()
             if node_to_idx[u] != node_to_idx[v]]  # exclude self-loops
    if len(edges) > 0:
        src = [e[0] for e in edges]
        dst = [e[1] for e in edges]
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        # Validate bounds
        assert edge_index.max() < N, f'edge_index out of bounds: max={edge_index.max()}, N={N}'
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)

    data = HeteroData()
    data['op'].x       = torch.tensor(x, dtype=torch.float)
    data['op'].y_slack = torch.tensor(slack_vals, dtype=torch.float)
    data['op'].y_step  = torch.tensor([step_time], dtype=torch.float)  # shape [1]
    data['op', 'depends_on', 'op'].edge_index = edge_index

    return data


print('HeteroData converter defined.')

## Trace Loader

Handles both Picotron `traceEvents` dicts and bare op-list JSON formats.

In [ ]:
def load_ops_from_trace(trace_path):
    """
    Load operations from a Picotron / Chakra JSON trace file.

    FIX (v2 final): handles both:
      - {'traceEvents': [...]} format (standard Chrome tracing)
      - bare list format [{'name':..., 'ts':..., 'dur':...}, ...]
    """
    with open(trace_path, 'r') as f:
        trace = json.load(f)

    # Normalize to a list of events
    if isinstance(trace, dict):
        events = trace.get('traceEvents', trace.get('events', []))
    elif isinstance(trace, list):
        events = trace
    else:
        return []

    ops = []
    for event in events:
        if not isinstance(event, dict):
            continue
        name  = event.get('name', event.get('op_type', event.get('op_name', 'unknown')))
        ts    = event.get('ts',   event.get('start_time',   event.get('start', None)))
        dur   = event.get('dur',  event.get('duration',     event.get('end', None)))

        # If 'end' was provided instead of 'dur', compute dur
        if dur is None and ts is not None and event.get('end') is not None:
            dur = float(event['end']) - float(ts)

        if ts is None or dur is None:
            continue
        ts  = float(ts)
        dur = float(dur)
        if dur <= 0:
            continue
        ops.append({'name': str(name), 'start_time': ts, 'duration': dur})

    return ops


print('Trace loader defined.')

## Synthetic Augmentation (Fallback)

If your traces are fundamentally single-stream (serial GPU kernels), even the best
pruning won't recover enough parallelism. This generates synthetic fork-join graphs
to supplement the dataset. Only use if `avg_cp_ratio > 0.80` after the fixes.

In [ ]:
def generate_synthetic_graph(n_nodes=None, n_parallel_groups=None, rng=None):
    """
    Generate a synthetic fork-join DAG with controlled parallelism.

    Structure: source → parallel group 1 → join → parallel group 2 → ... → sink
    Ensures meaningful branching factor and low CP ratio.

    Returns: list of op dicts (same format as load_ops_from_trace)
    """
    if rng is None:
        rng = np.random.default_rng()
    if n_nodes is None:
        n_nodes = rng.integers(15, 50)
    if n_parallel_groups is None:
        n_parallel_groups = rng.integers(2, 5)

    ops = []
    t   = 0.0
    group_size = max(2, n_nodes // (n_parallel_groups + 1))

    for group in range(n_parallel_groups):
        # Fork: multiple ops starting at same time with different durations
        group_start = t
        durations = rng.exponential(scale=10.0, size=group_size)
        for i, dur in enumerate(durations):
            # Add slight time jitter so parallel detection fires correctly
            jitter = rng.uniform(0.0, 0.5)
            ops.append({
                'name'      : f'g{group}_op{i}',
                'start_time': group_start + jitter,
                'duration'  : float(dur),
            })
        # Join: next group starts after max of current group
        t = group_start + max(durations) + rng.uniform(1.0, 3.0)

    # Add a few serial ops between groups to make some non-CP nodes
    for i in range(3):
        dur = rng.exponential(scale=5.0)
        ops.append({'name': f'serial_{i}', 'start_time': t, 'duration': float(dur)})
        t += dur + rng.uniform(0.5, 2.0)

    return ops


def augment_with_synthetic(hetero_graphs, target_count=500, rng_seed=42):
    """
    Add synthetic graphs until we reach target_count.
    Only called if real graph count is insufficient.
    """
    rng    = np.random.default_rng(rng_seed)
    random.seed(rng_seed)
    added  = 0
    trials = 0

    while len(hetero_graphs) < target_count and trials < target_count * 5:
        trials += 1
        ops = generate_synthetic_graph(rng=rng)
        ops = sorted(ops, key=lambda x: x['start_time'])
        G   = build_dag_from_ops(ops)
        if G is None:
            continue
        try:
            slack = compute_cpm_slack(G)
        except Exception:
            continue
        if compute_cp_ratio(G, slack) > CP_THRESHOLD:
            continue
        hetero_graphs.append(graph_to_heterodata(G, slack))
        added += 1

    print(f'Synthetic augmentation: added {added} graphs (total: {len(hetero_graphs)})')
    return hetero_graphs


print('Synthetic augmentation defined.')

## Main Processing Pipeline

In [ ]:
def process_all_traces(trace_dir):
    """
    Full pipeline: trace JSON files → filtered HeteroData graphs.
    """
    trace_files = [
        os.path.join(trace_dir, f)
        for f in os.listdir(trace_dir)
        if f.endswith('.json')
    ] if os.path.isdir(trace_dir) else []

    print(f'Found {len(trace_files)} trace files in {trace_dir}.')

    kept_graphs   = []  # (G, slack) pairs
    hetero_graphs = []

    stats = {'total': 0, 'too_small': 0, 'dag_failed': 0, 'high_cp': 0, 'kept': 0}

    for path in tqdm(trace_files):
        stats['total'] += 1
        try:
            ops = load_ops_from_trace(path)
        except Exception as e:
            stats['dag_failed'] += 1
            continue

        if len(ops) < MIN_NODES:
            stats['too_small'] += 1
            continue

        ops = sorted(ops, key=lambda x: x['start_time'])
        G   = build_dag_from_ops(ops)
        if G is None:
            stats['dag_failed'] += 1
            continue

        try:
            slack = compute_cpm_slack(G)
        except Exception:
            stats['dag_failed'] += 1
            continue

        # FIX 5: drop near-chain graphs
        if compute_cp_ratio(G, slack) > CP_THRESHOLD:
            stats['high_cp'] += 1
            continue

        kept_graphs.append((G, slack))
        hetero_graphs.append(graph_to_heterodata(G, slack))
        stats['kept'] += 1

    print('\n=== Processing Summary ===')
    for k, v in stats.items():
        print(f'  {k:15s}: {v}')

    return kept_graphs, hetero_graphs


print('Pipeline defined.')

In [ ]:
# ── Run pipeline ───────────────────────────────────────────────────────────────
kept_graphs, hetero_graphs = process_all_traces(TRACE_DIR)
print(f'\nFrom real traces: {len(hetero_graphs)} graphs.')

In [ ]:
# ── Quality report (run BEFORE training) ───────────────────────────────────────
if kept_graphs:
    summary = graph_quality_report(kept_graphs)

    # If avg_cp_ratio is too high or graph count is too low, augment synthetically
    if summary['avg_cp_ratio'] > CP_THRESHOLD or summary['n_graphs'] < 400:
        print('\n→ Augmenting with synthetic fork-join graphs...')
        hetero_graphs = augment_with_synthetic(hetero_graphs, target_count=500)
else:
    print('No real graphs found. Generating fully synthetic dataset...')
    hetero_graphs = augment_with_synthetic([], target_count=500)

In [ ]:
# ── Save dataset ───────────────────────────────────────────────────────────────
with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(hetero_graphs, f)

print(f'Saved {len(hetero_graphs)} graphs to {OUTPUT_PATH}')
print('Ready for nb3 (HeteroGAT training).')

## Drive Backup

In [ ]:
if IN_COLAB:
    import shutil
    import tempfile

    with tempfile.TemporaryDirectory() as tmpdir:
        export_dir = os.path.join(tmpdir, 'nb4_export')
        os.makedirs(export_dir, exist_ok=True)

        shutil.copy2(OUTPUT_PATH, os.path.join(export_dir, 'graphs_v2.pkl'))

        if os.path.exists(GRAPH_QUALITY_FIG_PATH):
            shutil.copy2(GRAPH_QUALITY_FIG_PATH, os.path.join(export_dir, 'graph_quality_v2.png'))
        if os.path.exists(SAMPLE_DAG_FIG_PATH):
            shutil.copy2(SAMPLE_DAG_FIG_PATH, os.path.join(export_dir, 'sample_dag_v2.png'))

        archive_base = os.path.join(tmpdir, 'nb4_data_backup')
        archive_path = shutil.make_archive(archive_base, 'zip', export_dir)
        shutil.copy2(archive_path, BACKUP_ZIP_PATH)

    print(f'Drive backup written to {BACKUP_ZIP_PATH}')
else:
    print('Drive backup skipped outside Colab. Files remain in the local workspace.')


## Sanity Check — Single Graph Visualization

In [ ]:
if kept_graphs:
    G_s, slack_s = kept_graphs[0]
    N, E = len(G_s.nodes()), len(G_s.edges())
    cp_r = compute_cp_ratio(G_s, slack_s)
    avg_s = np.mean(list(slack_s.values()))
    std_s = np.std(list(slack_s.values()))

    print(f'Sample graph: {N} nodes, {E} edges')
    print(f'  CP ratio       : {cp_r:.3f}  (want < {CP_THRESHOLD})')
    print(f'  Slack mean/std : {avg_s:.5f} / {std_s:.5f}')
    print(f'  Avg out-degree : {E/N:.2f}  (want > 1.30)')

    if N <= 50:
        plt.figure(figsize=(10, 6))
        pos        = nx.spring_layout(G_s, seed=42)
        cp_nodes   = [n for n, s in slack_s.items() if s < 1e-9]
        other_nodes= [n for n in G_s.nodes() if n not in cp_nodes]
        nx.draw_networkx_nodes(G_s, pos, nodelist=cp_nodes,    node_color='tomato',    node_size=400, label='critical')
        nx.draw_networkx_nodes(G_s, pos, nodelist=other_nodes, node_color='steelblue', node_size=400, label='off-critical')
        nx.draw_networkx_edges(G_s, pos, arrows=True, arrowsize=15)
        nx.draw_networkx_labels(G_s, pos, font_size=7)
        plt.legend(); plt.title(f'Sample DAG — CP ratio: {cp_r:.2f}'); plt.axis('off')
        plt.tight_layout()
        plt.savefig(SAMPLE_DAG_FIG_PATH, dpi=120)
        plt.show()

## Pre-Training Checklist

Before running nb3:

- [ ] `avg_cp_ratio < 0.80` — if not, lower `EPS_FRACTION` to 0.08–0.10
- [ ] `avg_branching_factor > 1.30` — if not, set `MAX_PARENTS=1` or increase `EDGE_DROP_PROB`
- [ ] `n_graphs >= 400` — if too few, synthetic augmentation was triggered automatically
- [ ] `graphs_v2.pkl` saved to the shared Drive folder and loadable by nb3

If traces are fundamentally serial (single GPU stream), augmentation will supplement.
The HeteroGAT model in nb3 will train on the combined real + synthetic dataset.